#### python贪吃蛇
- 面向对象设计（蛇、食物、游戏控制器）

- 键盘事件处理、碰撞检测

In [2]:
import pygame
import random
from enum import Enum

pygame 2.6.1 (SDL 2.28.4, Python 3.10.9)
Hello from the pygame community. https://www.pygame.org/contribute.html


In [3]:
# 方向枚举
# 定义移动方向
class Direction(Enum):
    UP = (0, -1)
    DOWN = (0, 1)
    LEFT = (-1, 0)
    RIGHT = (1, 0)

class Snake:
    """蛇类 - 管理蛇的移动、生长、转向"""
    def __init__(self, start_pos, initial_length=3):
        self.positions = []  # 蛇身所有位置的列表
        self.direction = Direction.RIGHT
        self.grow_flag = False  # 是否需要生长
        
        # 初始化蛇身
        for i in range(initial_length):
            x = start_pos[0] - i * 20  # 假设格子大小为20
            y = start_pos[1]
            self.positions.append((x, y))
    
    def move(self):
        """移动蛇"""
        head = self.positions[0]
        dx, dy = self.direction.value
        new_head = (head[0] + dx * 20, head[1] + dy * 20)
        
        # 插入新头部
        self.positions.insert(0, new_head)
        
        # 如果不生长，移除尾部
        if not self.grow_flag:
            self.positions.pop()
        else:
            self.grow_flag = False
    
    def change_direction(self, new_direction):
        """改变方向（不能直接掉头）"""
        # 防止直接掉头
        if (new_direction.value[0] * -1, new_direction.value[1] * -1) != self.direction.value:
            self.direction = new_direction
    
    def grow(self):
        """设置生长标志"""
        self.grow_flag = True
    
    def check_collision(self):
        """检查是否撞到自己"""
        head = self.positions[0]
        return head in self.positions[1:]
    
    def get_head(self):
        """获取蛇头位置"""
        return self.positions[0]

class Food:
    """食物类 - 管理食物的生成和位置"""
    def __init__(self, grid_size, screen_size):
        self.grid_size = grid_size
        self.screen_size = screen_size
        self.position = None
        self.generate()
    
    def generate(self, snake_positions=None):
        """随机生成食物（避开蛇身）"""
        while True:
            x = random.randint(0, (self.screen_size[0] - self.grid_size) // self.grid_size) * self.grid_size
            y = random.randint(0, (self.screen_size[1] - self.grid_size) // self.grid_size) * self.grid_size
            self.position = (x, y)
            
            # 如果提供了蛇的位置，确保食物不生成在蛇身上
            if snake_positions and self.position in snake_positions:
                continue
            break
    
    def is_eaten(self, snake_head):
        """检查是否被吃"""
        return self.position == snake_head

class GameController:
    """游戏控制器 - 管理游戏状态、渲染和事件处理"""
    def __init__(self, width=600, height=400):
        pygame.init()
        self.width = width
        self.height = height
        self.screen = pygame.display.set_mode((width, height))
        pygame.display.set_caption("贪吃蛇")
        self.clock = pygame.time.Clock()
        self.grid_size = 20  # 格子大小
        self.font = pygame.font.Font(None, 36)
        
        self.reset_game()
    
    def reset_game(self):
        """重置游戏"""
        start_pos = (self.width // 2, self.height // 2)
        self.snake = Snake(start_pos)
        self.food = Food(self.grid_size, (self.width, self.height))
        self.food.generate(self.snake.positions)
        self.score = 0
        self.game_over = False
        self.paused = False
    
    def handle_events(self):
        """处理键盘事件"""
        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                return False
            
            if event.type == pygame.KEYDOWN:
                if self.game_over:
                    if event.key == pygame.K_SPACE:
                        self.reset_game()
                    elif event.key == pygame.K_ESCAPE:
                        return False
                else:
                    if event.key == pygame.K_UP:
                        self.snake.change_direction(Direction.UP)
                    elif event.key == pygame.K_DOWN:
                        self.snake.change_direction(Direction.DOWN)
                    elif event.key == pygame.K_LEFT:
                        self.snake.change_direction(Direction.LEFT)
                    elif event.key == pygame.K_RIGHT:
                        self.snake.change_direction(Direction.RIGHT)
                    elif event.key == pygame.K_p:  # 暂停
                        self.paused = not self.paused
                    elif event.key == pygame.K_ESCAPE:
                        return False
        return True
    
    def update(self):
        """更新游戏状态"""
        if self.game_over or self.paused:
            return
        
        # 移动蛇
        self.snake.move()
        
        # 检查是否吃到食物
        if self.food.is_eaten(self.snake.get_head()):
            self.snake.grow()
            self.score += 10
            self.food.generate(self.snake.positions)
        
        # 碰撞检测
        head = self.snake.get_head()
        
        # 检查边界碰撞
        if (head[0] < 0 or head[0] >= self.width or 
            head[1] < 0 or head[1] >= self.height):
            self.game_over = True
        
        # 检查自身碰撞
        if self.snake.check_collision():
            self.game_over = True
    
    def draw(self):
        """渲染游戏画面"""
        self.screen.fill((0, 0, 0))  # 黑色背景
        
        # 绘制网格线（可选）
        self._draw_grid()
        
        # 绘制蛇
        for i, pos in enumerate(self.snake.positions):
            color = (0, 255, 0) if i == 0 else (0, 200, 0)  # 蛇头亮绿色
            pygame.draw.rect(self.screen, color, 
                           (pos[0], pos[1], self.grid_size - 2, self.grid_size - 2))
        
        # 绘制食物
        pygame.draw.rect(self.screen, (255, 0, 0), 
                        (self.food.position[0], self.food.position[1], 
                         self.grid_size - 2, self.grid_size - 2))
        
        # 显示分数
        score_text = self.font.render(f"Score: {self.score}", True, (255, 255, 255))
        self.screen.blit(score_text, (10, 10))
        
        # 游戏结束或暂停提示
        if self.game_over:
            text = self.font.render("Game Over! Press SPACE to restart", True, (255, 255, 255))
            text_rect = text.get_rect(center=(self.width // 2, self.height // 2))
            self.screen.blit(text, text_rect)
        elif self.paused:
            text = self.font.render("PAUSED", True, (255, 255, 255))
            text_rect = text.get_rect(center=(self.width // 2, self.height // 2))
            self.screen.blit(text, text_rect)
        
        pygame.display.flip()
    
    def _draw_grid(self):
        """绘制网格（可选）"""
        for x in range(0, self.width, self.grid_size):
            pygame.draw.line(self.screen, (40, 40, 40), (x, 0), (x, self.height))
        for y in range(0, self.height, self.grid_size):
            pygame.draw.line(self.screen, (40, 40, 40), (0, y), (self.width, y))
    
    def run(self):
        """游戏主循环"""
        running = True
        while running:
            running = self.handle_events()
            self.update()
            self.draw()
            self.clock.tick(10)  # 控制游戏速度（10帧/秒）
        
        pygame.quit()



In [4]:
# 主程序入口
game = GameController()
game.run()